# 52. A History of Solar Activity over Millennia — Implementation / 구현

**Paper**: Usoskin, I. G., *Living Reviews in Solar Physics* **14**:3 (2017). DOI: [10.1007/s41116-017-0006-9](https://doi.org/10.1007/s41116-017-0006-9)

---

## Purpose / 목적

**English.** This notebook reproduces the key analytical ideas of Usoskin (2017) using synthetic (but physically motivated) time series:
1. Generate a synthetic ¹⁴C / ¹⁰Be series combining the Hallstatt (~2400 yr), de Vries/Suess (~210 yr), Gleissberg (~90 yr), Schwabe (11 yr) cycles plus Grand Minima.
2. Perform power-spectral analysis to confirm the embedded periodicities.
3. Implement a Grand Minima detection algorithm based on a threshold of the smoothed SN series.
4. Reconstruct modulation potential $\phi$ from synthetic ¹⁰Be (toy inversion of the force-field + yield function).
5. Run superposed-epoch analysis around Grand Minima onsets.
6. Generate the 774/775 AD extreme event signature in $\Delta^{14}\text{C}$ using a simple two-box carbon model.

**한국어.** 이 노트북은 Usoskin (2017)의 핵심 분석 아이디어를 합성(그러나 물리적으로 동기 부여된) 시계열을 사용하여 재현한다:
1. Hallstatt (~2400년), de Vries/Suess (~210년), Gleissberg (~90년), Schwabe (11년) 주기 및 대극소기를 결합한 합성 ¹⁴C / ¹⁰Be 시리즈 생성.
2. 임베디드된 주기성을 확인하기 위한 파워 스펙트럼 분석.
3. 평활화된 SN 시리즈의 임계값에 기반한 대극소기 검출 알고리즘 구현.
4. 합성 ¹⁰Be에서 변조 퍼텐셜 $\phi$ 재구성 (force-field + 수율 함수의 장난감 역변환).
5. 대극소기 시작 주위의 중첩 시기(superposed epoch) 분석 실행.
6. 단순 2-박스 탄소 모델을 사용하여 $\Delta^{14}\text{C}$에서 774/775 AD 극한 사건 신호 생성.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

np.random.seed(20260423)
plt.rcParams.update({'figure.figsize': (11, 4), 'font.size': 10})

## 1. Synthetic Holocene SN Series / 합성 홀로세 SN 시리즈

**English.** We construct an 11,000-year sunspot-number-equivalent series as a superposition of quasi-periodic components plus Gaussian noise and randomly injected Grand Minima. The amplitudes roughly match the Usoskin et al. (2014, 2016a) reconstruction (moderate baseline SN ~ 40, Grand Minima depth to SN ~ 5).

**한국어.** 준주기 성분과 가우시안 잡음 및 무작위로 주입된 대극소기의 중첩으로 11,000년 흑점 수 등가 시리즈를 구성한다. 진폭은 Usoskin et al. (2014, 2016a) 재구성(중간 기준선 SN ~ 40, 대극소기 깊이 SN ~ 5)과 대략 일치한다.

In [ ]:
def synthetic_holocene_sn(years: np.ndarray,
                          baseline: float = 40.0,
                          amp_hallstatt: float = 7.0,
                          amp_gleissberg: float = 6.0,
                          amp_suess: float = 4.0,
                          amp_schwabe: float = 25.0,
                          noise_sigma: float = 5.0,
                          n_minima: int = 25,
                          rng: np.random.Generator | None = None) -> np.ndarray:
    """Generate a synthetic SN-equivalent time series over the Holocene.

    Args:
        years: Annual year array (e.g., arange(-9000, 2000)).
        baseline: Mean moderate activity level in SN units.
        amp_hallstatt: Amplitude of the ~2400-yr Hallstatt cycle.
        amp_gleissberg: Amplitude of the ~90-yr Gleissberg cycle.
        amp_suess: Amplitude of the ~210-yr de Vries/Suess cycle.
        amp_schwabe: Amplitude of the ~11-yr Schwabe cycle.
        noise_sigma: Standard deviation of additive Gaussian noise.
        n_minima: Number of Grand Minima to randomly inject.
        rng: Optional numpy Generator for reproducibility.

    Returns:
        ndarray: The synthetic SN series of the same length as years.
    """
    if rng is None:
        rng = np.random.default_rng(20260423)
    t = years.astype(float)
    sn = (baseline
          + amp_hallstatt * np.sin(2 * np.pi * t / 2400.0)
          + amp_suess * np.sin(2 * np.pi * t / 210.0 + 1.1)
          + amp_gleissberg * np.sin(2 * np.pi * t / 90.0 + 0.3)
          + amp_schwabe * np.abs(np.sin(np.pi * t / 11.0))
          + rng.normal(0.0, noise_sigma, size=t.size))
    # Inject Grand Minima — bimodal duration distribution (Maunder 30-90 yr, Sporer 100-160 yr).
    centers = rng.choice(t, size=n_minima, replace=False)
    for c in centers:
        duration = rng.choice([50.0, 70.0, 80.0, 130.0, 160.0],
                              p=[0.25, 0.25, 0.20, 0.15, 0.15])
        mask = np.abs(t - c) < duration / 2.0
        # Suppression factor: SN drops from baseline to ~5-10.
        sn[mask] *= 0.15
    return np.clip(sn, 0.0, None)


years = np.arange(-9000, 2000)
sn_series = synthetic_holocene_sn(years)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(years, sn_series, color='black', lw=0.5)
ax.set_xlabel('Year (BC/AD)')
ax.set_ylabel('Synthetic SN')
ax.set_title('Synthetic Holocene solar activity series / 합성 홀로세 태양 활동 시리즈')
plt.tight_layout()
plt.show()

## 2. Power Spectrum — Hallstatt, Gleissberg, Schwabe / 파워 스펙트럼

**English.** Compute the Lomb-Scargle-free FFT power spectrum of the synthetic series (after removing the mean). Expect peaks at frequencies $1/2400$, $1/210$, $1/90$, and $1/11$ yr⁻¹.

**한국어.** 합성 시리즈의 파워 스펙트럼(평균 제거 후 FFT)을 계산한다. 주파수 $1/2400$, $1/210$, $1/90$, $1/11$ yr⁻¹에서 피크를 기대한다.

In [ ]:
def power_spectrum(x: np.ndarray, dt: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Compute a one-sided FFT power spectrum (simple periodogram).

    Args:
        x: Input time series (1D array).
        dt: Sample spacing in years.

    Returns:
        Tuple of (frequency array [1/yr], power array).
    """
    x0 = x - np.mean(x)
    n = x0.size
    freqs = np.fft.rfftfreq(n, d=dt)
    power = np.abs(np.fft.rfft(x0)) ** 2 / n
    return freqs, power


freqs, power = power_spectrum(sn_series)
periods = 1.0 / np.where(freqs > 0, freqs, np.nan)

fig, ax = plt.subplots(figsize=(11, 4))
ax.loglog(periods, power, color='navy', lw=0.7)
for p, label in [(11, 'Schwabe ~11 yr'), (90, 'Gleissberg ~90 yr'),
                 (210, 'de Vries/Suess ~210 yr'), (2400, 'Hallstatt ~2400 yr')]:
    ax.axvline(p, color='red', ls='--', alpha=0.5)
    ax.text(p, power.max() * 0.7, label, rotation=90, ha='right', va='top', fontsize=8)
ax.set_xlabel('Period (yr)')
ax.set_ylabel('Power (arb. units)')
ax.set_xlim(5, 5000)
ax.set_title('Power spectrum of synthetic series / 합성 시리즈 파워 스펙트럼')
plt.tight_layout()
plt.show()

## 3. Grand Minima Detection / 대극소기 검출

**English.** Usoskin et al. (2007, 2014) define Grand Minima as periods where the decadal-smoothed SN falls below a threshold (~SN 15–20) for at least ~30 yr. We implement a simple detector and report the total time fraction in Grand Minima, comparing to the paper's ~17%.

**한국어.** Usoskin et al. (2007, 2014)은 대극소기를 10년 단위 평활화 SN이 임계값(~SN 15–20) 아래로 최소 ~30년 동안 떨어지는 기간으로 정의한다. 단순 검출기를 구현하고 대극소기의 총 시간 분율을 보고하며, 논문의 ~17%와 비교한다.

In [ ]:
def detect_grand_minima(years: np.ndarray,
                        sn: np.ndarray,
                        smoothing_window: int = 11,
                        threshold: float = 15.0,
                        min_duration: int = 30) -> list[tuple[float, float, int]]:
    """Detect Grand Minima as intervals where smoothed SN falls below a threshold.

    Args:
        years: Year array (sorted ascending).
        sn: SN time series aligned with years.
        smoothing_window: Boxcar window length (years) used for smoothing.
        threshold: SN threshold below which an interval qualifies.
        min_duration: Minimum duration in years to count as a Grand Minimum.

    Returns:
        List of tuples (start_year, end_year, duration_years).
    """
    kernel = np.ones(smoothing_window) / smoothing_window
    smoothed = np.convolve(sn, kernel, mode='same')
    below = smoothed < threshold
    minima = []
    in_min = False
    start = 0
    for i, b in enumerate(below):
        if b and not in_min:
            in_min = True
            start = i
        elif not b and in_min:
            in_min = False
            duration = i - start
            if duration >= min_duration:
                minima.append((float(years[start]), float(years[i - 1]), int(duration)))
    if in_min:
        duration = below.size - start
        if duration >= min_duration:
            minima.append((float(years[start]), float(years[-1]), int(duration)))
    return minima


minima = detect_grand_minima(years, sn_series)
total_minima_years = sum(m[2] for m in minima)
fraction = total_minima_years / years.size
print(f'Detected {len(minima)} Grand Minima.')
print(f'Total duration: {total_minima_years} yr = {100 * fraction:.1f}% of the Holocene.')
print('(Usoskin 2017 reports ~17% = ~1/6 / Usoskin 2017은 ~17% = ~1/6 보고)')
print('\nFirst five detected minima (start, end, duration_yr):')
for m in minima[:5]:
    print(m)

## 4. Modulation Potential $\phi$ from ¹⁰Be / ¹⁰Be로부터 변조 퍼텐셜 재구성

**English.** A toy physics-based reconstruction. We use a simplified relation between ¹⁰Be production rate $Q$ and the modulation potential $\phi$ (fit to Kovaltsov et al. 2012 yield function at fixed $M$):
$$Q(\phi) \approx Q_0 \cdot \exp(-\phi/\phi_c)$$
where $\phi_c \approx 900$ MV is a scale parameter. Inverting: $\phi = -\phi_c \ln(Q/Q_0)$.

We map the SN series to a $Q$ series (higher activity → higher $\phi$ → lower $Q$) and then invert.

**한국어.** 장난감 물리 기반 재구성. 고정된 $M$에서 Kovaltsov et al. (2012) 수율 함수에 적합한 ¹⁰Be 생성률 $Q$와 변조 퍼텐셜 $\phi$의 단순화된 관계를 사용한다:
$$Q(\phi) \approx Q_0 \cdot \exp(-\phi/\phi_c)$$
여기서 $\phi_c \approx 900$ MV는 스케일 매개변수이다. 역변환: $\phi = -\phi_c \ln(Q/Q_0)$.

SN 시리즈를 $Q$ 시리즈에 매핑한(활동 증가 → $\phi$ 증가 → $Q$ 감소) 다음 역변환한다.

In [ ]:
def sn_to_modulation(sn: np.ndarray,
                     phi_baseline: float = 400.0,
                     phi_per_sn: float = 6.0) -> np.ndarray:
    """Map SN to modulation potential phi using a linear relation.

    Args:
        sn: SN series.
        phi_baseline: Modulation potential at SN = 0 (MV).
        phi_per_sn: Slope of phi vs SN (MV per SN unit).

    Returns:
        Modulation potential series in MV.
    """
    return phi_baseline + phi_per_sn * sn


def phi_to_be10(phi_mv: np.ndarray,
                q0: float = 0.04,
                phi_c: float = 900.0) -> np.ndarray:
    """Convert modulation potential to ^{10}Be production via toy exponential model.

    Args:
        phi_mv: Modulation potential array in MV.
        q0: Zero-modulation production rate (atoms cm^-2 s^-1).
        phi_c: Scale parameter in MV.

    Returns:
        ^{10}Be production rate array.
    """
    return q0 * np.exp(-phi_mv / phi_c)


def be10_to_phi(q: np.ndarray,
                q0: float = 0.04,
                phi_c: float = 900.0) -> np.ndarray:
    """Invert the exponential phi-Q relation.

    Args:
        q: ^{10}Be production rate array.
        q0: Zero-modulation production rate.
        phi_c: Scale parameter in MV.

    Returns:
        Reconstructed modulation potential (MV).
    """
    return -phi_c * np.log(q / q0)


phi_true = sn_to_modulation(sn_series)
be10 = phi_to_be10(phi_true)
# Add 5% measurement noise.
be10_noisy = be10 * (1.0 + 0.05 * np.random.default_rng(42).standard_normal(be10.size))
phi_recon = be10_to_phi(be10_noisy)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(years, be10_noisy * 1e4, color='teal', lw=0.6)
axes[0].set_ylabel(r'$^{10}$Be prod. (10$^{-4}$ at cm$^{-2}$ s$^{-1}$)')
axes[0].set_title('Synthetic noisy ¹⁰Be production / 합성 잡음 ¹⁰Be 생성')
axes[1].plot(years, phi_true, color='black', lw=0.6, label='True')
axes[1].plot(years, phi_recon, color='red', lw=0.4, alpha=0.7, label='Reconstructed')
axes[1].set_xlabel('Year (BC/AD)')
axes[1].set_ylabel(r'$\phi$ (MV)')
axes[1].legend()
plt.tight_layout()
plt.show()

err = np.std(phi_recon - phi_true)
print(f'RMS reconstruction error in phi: {err:.1f} MV')

## 5. Superposed Epoch Analysis around Grand Minima / 대극소기 주위 중첩 시기 분석

**English.** Align the SN series on detected Grand Minima onsets to reveal the typical signature (depth, duration, recovery profile). This reproduces the analysis style used by Usoskin et al. for clustering and Hallstatt-cycle studies.

**한국어.** 검출된 대극소기 시작에 SN 시리즈를 정렬하여 전형적인 신호(깊이, 지속 기간, 회복 프로파일)를 밝힌다. 이는 군집화 및 Hallstatt 주기 연구에 Usoskin et al.이 사용한 분석 스타일을 재현한다.

In [ ]:
def superposed_epoch(years: np.ndarray,
                     sn: np.ndarray,
                     onsets: list[float],
                     window: tuple[int, int] = (-100, 300)) -> tuple[np.ndarray, np.ndarray]:
    """Compute superposed-epoch mean and std around event onsets.

    Args:
        years: Year array.
        sn: SN series.
        onsets: List of onset years (must lie within years array).
        window: (before, after) offsets in years.

    Returns:
        Tuple (lag array, stack array of shape (n_events, n_lag)).
    """
    lags = np.arange(window[0], window[1])
    stack = []
    for onset in onsets:
        idx0 = int(np.searchsorted(years, onset))
        if idx0 + window[1] > sn.size or idx0 + window[0] < 0:
            continue
        stack.append(sn[idx0 + window[0]: idx0 + window[1]])
    return lags, np.array(stack)


onsets = [m[0] for m in minima]
lags, stack = superposed_epoch(years, sn_series, onsets)
mean = stack.mean(axis=0)
std = stack.std(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(lags, mean - std, mean + std, color='orange', alpha=0.3, label=r'$\pm 1\sigma$')
ax.plot(lags, mean, color='darkred', lw=2, label='Mean over %d events' % stack.shape[0])
ax.axvline(0, color='black', ls='--', lw=0.8)
ax.set_xlabel('Lag from Grand Minimum onset (yr)')
ax.set_ylabel('SN')
ax.set_title('Superposed epoch around Grand Minima / 대극소기 주변 중첩 시기')
ax.legend()
plt.tight_layout()
plt.show()

## 6. The 774/775 AD Extreme Event Signature / 774/775 AD 극한 사건 신호

**English.** Following Miyake et al. (2012) and Usoskin et al. (2013), we model the 775 AD event as an instantaneous stratospheric injection of ¹⁴C. Using a two-box (atmosphere/biosphere) model, a 1-yr impulse $\delta Q$ produces a sharp spike followed by an exponential decay with a characteristic carbon-cycle timescale $\tau \sim 20{-}25$ yr (effective).

**한국어.** Miyake et al. (2012)과 Usoskin et al. (2013)을 따라, 775 AD 사건을 ¹⁴C의 순간적인 성층권 주입으로 모델링한다. 2-박스(대기/생물권) 모델을 사용하여 1년 펄스 $\delta Q$는 특성 탄소 순환 시간 척도 $\tau \sim 20{-}25$년(유효)을 갖는 지수 붕괴가 뒤따르는 날카로운 스파이크를 생성한다.

In [ ]:
def ad775_event_signature(t_yr: np.ndarray,
                          onset: float = 775.0,
                          amplitude_permil: float = 12.0,
                          tau_yr: float = 22.0) -> np.ndarray:
    """Model the Delta^14C response to an impulsive stratospheric ^14C injection.

    Args:
        t_yr: Time axis in years AD.
        onset: Year of the instantaneous injection.
        amplitude_permil: Peak deviation from baseline in per mille.
        tau_yr: Effective carbon-cycle decay timescale.

    Returns:
        Delta^14C perturbation series (per mille).
    """
    # Step-up lasting 1 yr, then exponential decay.
    dt = t_yr - onset
    delta = np.where(dt >= 0,
                     amplitude_permil * np.exp(-dt / tau_yr),
                     0.0)
    # A small pre-ramp to mimic finite biennial measurement precision.
    pre = (dt >= -1) & (dt < 0)
    delta[pre] = amplitude_permil * (dt[pre] + 1.0)
    return delta


t_ad = np.arange(760.0, 800.0, 0.5)
baseline = -8.0  # approximate IntCal Delta^14C level around 775 AD (permil)
noise = 0.5 * np.random.default_rng(7).standard_normal(t_ad.size)
delta14c = baseline + ad775_event_signature(t_ad) + noise

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_ad, delta14c, marker='o', ms=3, lw=1.0, color='darkgreen',
        label='Modeled biennial Δ¹⁴C / 모델 격년 Δ¹⁴C')
ax.axvline(775, color='red', ls='--', alpha=0.6, label='775 AD event / 775 AD 사건')
ax.set_xlabel('Year AD')
ax.set_ylabel(r'$\Delta^{14}$C (‰)')
ax.set_title('774/775 AD extreme event signature / 774/775 AD 극한 사건 신호')
ax.legend()
plt.tight_layout()
plt.show()

peak = np.max(delta14c) - baseline
print(f'Modeled peak Delta^14C rise: {peak:.1f} permil (Miyake et al. 2012 reported ~12 permil).')

## 7. Summary / 요약

**English.** We have reproduced — at a conceptual and qualitative level — six key elements of Usoskin (2017):
1. The coexistence of the Schwabe (11 yr), Gleissberg (~90 yr), de Vries/Suess (~210 yr) and Hallstatt (~2400 yr) quasi-periodicities in a single cosmogenic-proxy-equivalent time series.
2. Power-spectrum identification of these periodicities.
3. A Grand Minima detection algorithm yielding a time fraction consistent with the paper's ~17%.
4. A physics-based toy inversion from ¹⁰Be production rate to modulation potential $\phi$.
5. Superposed-epoch analysis revealing the characteristic Grand Minimum depth and recovery profile.
6. The 775 AD extreme-SEP event signature in $\Delta^{14}\text{C}$, dominated by a sharp impulse plus exponential carbon-cycle decay.

**한국어.** 우리는 개념적/정성적 수준에서 Usoskin (2017)의 여섯 가지 핵심 요소를 재현했다:
1. 단일 우주선 기원 프록시 등가 시계열에서 Schwabe (11년), Gleissberg (~90년), de Vries/Suess (~210년), Hallstatt (~2400년) 준주기성의 공존.
2. 이러한 주기성의 파워 스펙트럼 식별.
3. 논문의 ~17%와 일치하는 시간 분율을 산출하는 대극소기 검출 알고리즘.
4. ¹⁰Be 생성률에서 변조 퍼텐셜 $\phi$로의 물리 기반 장난감 역변환.
5. 특성적 대극소기 깊이 및 회복 프로파일을 드러내는 중첩 시기 분석.
6. $\Delta^{14}\text{C}$의 775 AD 극한 SEP 사건 신호, 날카로운 임펄스와 지수 탄소 순환 붕괴 지배.

The synthetic approach skips the full chain (archive → $Q$ → $\phi$ → $F_o$ → SN) — for a full implementation one would need the Bern3D-LPJ carbon-cycle inversion, paleomagnetic VADM reconstructions (GMAG.9k), and Kovaltsov et al. (2012) yield tables. These are the professional production tools referenced by Usoskin (2017).

합성 접근은 전체 사슬(아카이브 → $Q$ → $\phi$ → $F_o$ → SN)을 건너뛴다 — 전체 구현에는 Bern3D-LPJ 탄소 순환 역변환, 고지자기 VADM 재구성 (GMAG.9k), Kovaltsov et al. (2012) 수율 테이블이 필요하다. 이들이 Usoskin (2017)이 참조하는 전문 제작 도구이다.